# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the daily performance of one content item for one client on one report date.

Time Window:
March 2026 (month = "2026-03")

In [16]:
!pip -q install duckdb datasets

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- client_has_ga4

## Label
Predict pages likely to decline in search performance.

## Context
- report_date
- client_hash_id
- content_hash_id

## Excluded
- June 2026 data
- Future information
- Label-derived columns

Reason:
These are excluded to avoid data leakage.

In [17]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

print("✅ Hugging Face connected successfully")

✅ Hugging Face connected successfully


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
dataset = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT *
FROM read_parquet(
'{dataset}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5;
"""

con.sql(query).df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [19]:
query = f"""
SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT
client_hash_id || content_hash_id || CAST(report_date AS VARCHAR)
) AS unique_rows
FROM read_parquet(
'{dataset}/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


In [20]:
query = f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS first_date,
MAX(report_date) AS last_date
FROM read_parquet(
'{dataset}/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [21]:
query = f"""
SELECT
COUNT(*) AS available_rows
FROM read_parquet(
'{dataset}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


In [22]:
query = f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_sum_position,
client_has_gsc,
client_has_ga4
FROM read_parquet(
'{dataset}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10;
"""

con.sql(query).df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,True,False
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,True,False
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,True,False
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,True,False
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,True,False
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,1756,True,False
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,1496,True,False
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,180,True,False
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,434,True,False
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,9,True,False


## Feature Availability

1. **gsc_impressions** – Knowable at the decision moment because they come from historical Search Console data.

2. **gsc_clicks** – Knowable because they are historical click data available before prediction.

3. **gsc_sum_position** – Knowable because historical search position is already available.

4. **client_has_gsc** – Knowable because it is client metadata available before prediction.

5. **client_has_ga4** – Knowable because it is client metadata available before prediction.

In [23]:
query = f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
gsc_clicks,
gsc_impressions,
gsc_sum_position,
gsc_clicks AS leaked_label
FROM read_parquet(
'{dataset}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10;
"""

con.sql(query).df()

,report_date,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_sum_position,leaked_label
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,0,20,67,0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,0,1,0,0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,1,125,616,1
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,0,7,28,0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0,11,25,0
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,1,239,1756,1
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,0,191,1496,0
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,0,55,180,0
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,0,77,434,0
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,0,2,9,0


## Leakage Experiment

I intentionally created a label-derived column (`leaked_label`) by copying the click values.

This feature would cause data leakage because it contains information that would not be available when making predictions.

The leaked column should be removed before training the model.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset contains historical warehouse data only.

It cannot explain why rankings changed or capture external factors such as Google algorithm updates, competitor actions, or seasonality.

The dataset has an unbalanced history because different clients started providing data at different times.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.